# AuNP Protein Corona Proteomics — Healthy vs Cancer/Cachexia Rat Serum

This notebook analyzes **rat serum-derived AuNP protein coronas only**: RS (healthy rat serum) versus RC (cancer/cachexia rat serum). Human proteomics input and analyses from the exploratory source notebook have been excluded.

The workflow retains the original analysis families: technical-injection averaging, PCA, protein-abundance distributions, row Z-score heatmaps, volcano plot, **tornado plot of volcano-selected proteins**, **relative abundance of selected proteins**, and independent filtering strategies.

**Important:** The original notebook uses two different source tables: a protein-measurements export and a relative-abundance table containing `RS`, `RC`, `RS %`, and `RC %`. Select both if available. The original thresholds and calculations are described in the corresponding sections; no statistical significance is inferred when source p-values are missing.

## 1. Configuration and input selection

Run the file dialogs to select the rat protein-measurements CSV and (optionally) the relative-abundance CSV. Set the paths manually if dialogs are unavailable. Results are saved in a dedicated folder.

In [ ]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

MEASUREMENTS_FILE = None  # e.g. Path("data/rat_protein_measurements.csv")
RELATIVE_ABUNDANCE_FILE = None  # e.g. Path("data/rat_relative_abundance.csv")
OUTPUT_DIR = Path("results/rat_AuNP_corona_proteomics")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RS_TOKEN, RC_TOKEN = "_RS", "_RC"
P_THRESHOLD = 0.05
FC_THRESHOLD = 1.5
VOLCANO_TOP_LABELS = 10
TORNADO_TOP_PER_GROUP = 10
TOP_RELATIVE_PROTEINS = 15

def choose_csv(title):
    try:
        import tkinter as tk
        from tkinter import filedialog
        window = tk.Tk()
        window.withdraw()
        window.attributes("-topmost", True)
        filename = filedialog.askopenfilename(
            title=title, filetypes=[("CSV files", "*.csv")]
        )
        window.destroy()
    except Exception as exc:
        raise RuntimeError("File dialog unavailable; set the path in configuration.") from exc
    if not filename:
        raise ValueError("No file selected.")
    return Path(filename)

def read_export(path):
    # Some vendor exports contain two metadata rows above the actual header.
    for skip in (0, 2):
        frame = pd.read_csv(path, skiprows=skip, low_memory=False)
        frame.columns = frame.columns.astype(str).str.strip()
        if "Description" in frame.columns or "Accession" in frame.columns:
            return frame
    raise ValueError("Expected 'Description' or 'Accession' in the proteomics export.")

measurements_path = Path(MEASUREMENTS_FILE) if MEASUREMENTS_FILE else choose_csv(
    "Select rat serum protein-measurements CSV (RS vs RC)"
)
measurements = read_export(measurements_path)
print("Measurements:", measurements.shape, "|", measurements_path)

## 2. Identify individual RS/RC injections — no averaging

Keep every `_i1` and `_i2` column independently. The supplied naming pattern contains 24 injection columns (12 RS and 12 RC). No averaging or collapsing of injections is performed for PCA or heatmap. Group means are computed later only for descriptive abundance comparisons and fold change.

In [ ]:
def sample_group(name):
    """Recognize RS1, RS2, RC1, RC2, etc. in exported column names."""
    match = re.search(r"(?:^|_)(RS|RC)\d+(?=_|$)", str(name))
    return match.group(1) if match else None

# Recognize explicit injection suffixes (_i1, _i2, ...) and optional
# pandas-generated duplicate-column suffixes (.1, .2, ...).
injection_pattern = re.compile(r"_i\d+(?:\.\d+)?$")
sample_cols = [
    col for col in measurements.columns
    if sample_group(col) is not None and injection_pattern.search(str(col))
]
if not sample_cols:
    raise ValueError(
        "No RS/RC injection columns found. Inspect measurements.columns "
        "and adjust the sample-identification rule if needed."
    )

expression = measurements[sample_cols].apply(pd.to_numeric, errors="coerce")

# Preserve each injection independently; DO NOT average _i1 and _i2.
individual = expression.copy()
groups = pd.Series({col: sample_group(col) for col in individual.columns})
if groups.isna().any() or set(groups.unique()) != {"RS", "RC"}:
    raise ValueError("Could not identify both RS and RC injection groups.")
print("Individual injection columns:", len(individual.columns))
print("RS / RC:", groups.value_counts().to_dict())
print("Injection names:", individual.columns.tolist())

if "Accession" in measurements.columns:
    protein_id_col = "Accession"
elif "Description" in measurements.columns:
    protein_id_col = "Description"
else:
    protein_id_col = None
protein_labels = (
    measurements[protein_id_col].astype(str)
    if protein_id_col is not None
    else pd.Series(measurements.index.astype(str), index=measurements.index)
)


## 3. Principal component analysis (PCA) — individual injections, no averaging

Each original injection column is an independent point in this exploratory visualization. For the supplied naming pattern, expect 24 points (12 RS + 12 RC). These technical injections should not be interpreted as independent biological replicates for hypothesis testing.

In [ ]:
# PCA observations are the individual injections, never group means.
assert individual.shape[1] == 24, f"PCA requires 24 injections; found {individual.shape[1]}"
assert (groups == "RS").sum() == 12 and (groups == "RC").sum() == 12
minimum_present = max(1, int(np.ceil(len(individual.columns) * 0.5)))
pca_matrix = individual.dropna(thresh=minimum_present).copy()
pca_matrix = pca_matrix.loc[pca_matrix.var(axis=1, skipna=True).fillna(0) > 0]
pca_matrix = pca_matrix.T
pca_matrix = pca_matrix.fillna(pca_matrix.median(axis=0)).fillna(0)
if min(pca_matrix.shape) < 2:
    raise ValueError("At least two samples and two valid proteins are required for PCA.")
scaled = StandardScaler().fit_transform(pca_matrix)
model = PCA(n_components=2)
scores = model.fit_transform(scaled)
pca_scores = pd.DataFrame(scores, columns=["PC1", "PC2"], index=pca_matrix.index)
pca_scores["Group"] = groups.loc[pca_scores.index].to_numpy()
assert len(pca_scores) == 24, "PCA must have 24 distinct observations."
print("PCA observations:", len(pca_scores), "|", pca_scores["Group"].value_counts().to_dict())
pca_scores.to_csv(OUTPUT_DIR / "pca_scores.csv", index_label="Sample")
loadings = pd.DataFrame(
    model.components_.T, columns=["PC1", "PC2"],
    index=protein_labels.loc[pca_matrix.columns].to_numpy()
)
loadings.to_csv(OUTPUT_DIR / "pca_loadings.csv", index_label="Protein")
fig, ax = plt.subplots(figsize=(8, 6))
sns.scatterplot(data=pca_scores, x="PC1", y="PC2", hue="Group", s=110, ax=ax)
ax.set_xlabel(f"PC1 ({model.explained_variance_ratio_[0]*100:.1f}%)")
ax.set_ylabel(f"PC2 ({model.explained_variance_ratio_[1]*100:.1f}%)")
ax.set_title("AuNP corona proteomics — 24 individual injections")
fig.tight_layout()
for ext in ("png", "svg"):
    fig.savefig(OUTPUT_DIR / f"pca_RS_RC.{ext}", dpi=300)
plt.show()

## 4. Protein abundance and differential-abundance table

`Mean_RS` and `Mean_RC` are computed from the original individual injection columns **only for descriptive group-level comparisons** (fold change, volcano and abundance tornado). They are never used as PCA points or heatmap columns.

In [ ]:
analysis = measurements.copy()
analysis["Mean_RS"] = individual.loc[:, groups[groups == "RS"].index].mean(axis=1)
analysis["Mean_RC"] = individual.loc[:, groups[groups == "RC"].index].mean(axis=1)
analysis["Description_clean"] = (
    analysis["Description"].astype(str).str.split("OS=", regex=False).str[0].str.strip()
    if "Description" in analysis else protein_labels
)
analysis["log2FC"] = np.where(
    (analysis["Mean_RS"] > 0) & (analysis["Mean_RC"] > 0),
    np.log2(analysis["Mean_RC"].where(analysis["Mean_RC"] > 0) /
            analysis["Mean_RS"].where(analysis["Mean_RS"] > 0)),
    np.nan
)
for column in ("Anova (p)", "q Value", "Intensity"):
    if column in analysis:
        analysis[column] = pd.to_numeric(analysis[column], errors="coerce")
if "Anova (p)" not in analysis:
    print("WARNING: source 'Anova (p)' absent; statistical volcano selection unavailable.")
analysis.to_csv(OUTPUT_DIR / "protein_abundance_RS_RC.csv", index=False)

## 5. Heatmap — original individual injection columns only

Show every original `_i1` and `_i2` column separately: 12 RS and 12 RC (24 columns in total), without any group-mean or injection-mean columns. Row Z-scores use the individual measurements.

In [ ]:
def row_zscore(matrix):
    values = matrix.astype(float)
    means = values.mean(axis=1)
    sds = values.std(axis=1, ddof=0).replace(0, np.nan)
    return values.sub(means, axis=0).div(sds, axis=0)

# Exactly 24 injections: 12 healthy-serum (RS), then 12 cancer/cachexia (RC).
ordered_samples = groups[groups == "RS"].index.tolist() + groups[groups == "RC"].index.tolist()
if len(ordered_samples) != 24 or (groups == "RS").sum() != 12 or (groups == "RC").sum() != 12:
    raise ValueError(f"Expected 12 RS + 12 RC injections; found {groups.value_counts().to_dict()}.")

# Plot ONLY the 24 individual injection columns; never append group means.
heat = individual.loc[:, ordered_samples].copy()
assert heat.shape[1] == 24, "Heatmap must contain exactly 24 injection columns."
assert not any(str(col).lower().startswith("mean") for col in heat.columns), "Unexpected mean column."
if "Anova (p)" in analysis.columns:
    pvalues = pd.to_numeric(analysis["Anova (p)"], errors="coerce")
    top_idx = pvalues.dropna().nsmallest(100).index
    heat = heat.reindex(top_idx)
else:
    heat = heat.iloc[:100]

heat = heat.dropna(how="all")
# Impute per-protein (per-row) medians; do not fill using column medians.
heat = heat.T.fillna(heat.median(axis=1)).T
heat_z = row_zscore(heat).dropna(how="any")
heat_z.index = protein_labels.loc[heat_z.index].astype(str).to_numpy()
heat_z.index.name = "Protein"

if heat_z.empty:
    raise ValueError("No nonconstant protein profiles available for the heatmap.")

print(f"Heatmap: {heat_z.shape[0]} proteins x {heat_z.shape[1]} injections")
print("Sample columns:", heat_z.columns.tolist())
fig, ax = plt.subplots(figsize=(12, 13))
sns.heatmap(
    heat_z, cmap="RdBu_r", center=0, vmin=-2, vmax=2,
    yticklabels=False, xticklabels=True, ax=ax,
    cbar_kws={"label": "Row Z-score"}
)
ax.axvline(12, color="black", linewidth=1.5)
ax.set_title("Protein abundance — 24 injections (12 RS + 12 RC)")
ax.set_xlabel("Individual samples (RS and RC, each injection shown separately)")
ax.tick_params(axis="x", labelrotation=65, labelsize=9)
fig.tight_layout()
for ext in ("png", "svg"):
    fig.savefig(OUTPUT_DIR / f"heatmap_24_individual_injections.{ext}", dpi=300, bbox_inches="tight")
plt.show()
heat_z.to_csv(OUTPUT_DIR / "heatmap_24_individual_injections_data.csv")

## 6. Volcano plot and selected proteins

The original volcano threshold is `Anova (p) < 0.05` and `|log2FC| ≥ log2(1.5)`. The ten most significant selected proteins are labeled. Selection is exported and reused directly in the tornado plot below; this avoids selecting unrelated proteins.

In [ ]:
if "Anova (p)" not in analysis:
    raise ValueError("The volcano plot requires source 'Anova (p)' values.")
analysis["Volcano_Significant"] = (
    (analysis["Anova (p)"] < P_THRESHOLD)
    & (analysis["log2FC"].abs() >= np.log2(FC_THRESHOLD))
)
selected = analysis.loc[analysis["Volcano_Significant"]].copy()
selected.to_csv(OUTPUT_DIR / "volcano_selected_proteins.csv", index=False)
plot_data = analysis.loc[
    analysis["Anova (p)"].gt(0) & analysis["log2FC"].notna()
].copy()
fig, ax = plt.subplots(figsize=(9, 7))
colors = np.where(plot_data["Volcano_Significant"], "#cc3344", "#999999")
ax.scatter(plot_data["log2FC"], -np.log10(plot_data["Anova (p)"]),
           c=colors, s=28, alpha=0.75)
ax.axhline(-np.log10(P_THRESHOLD), color="steelblue", linestyle="--")
for sign in (-1, 1):
    ax.axvline(sign*np.log2(FC_THRESHOLD), color="forestgreen", linestyle="--")
for _, row in selected.nsmallest(VOLCANO_TOP_LABELS, "Anova (p)").iterrows():
    if row["Anova (p)"] > 0:
        ax.annotate(str(row["Description_clean"])[:45],
                    (row["log2FC"], -np.log10(row["Anova (p)"])), fontsize=7)
ax.set(xlabel="log2 fold change (RC / RS)", ylabel="-log10(ANOVA p-value)",
       title="AuNP corona differential protein abundance")
fig.tight_layout()
for ext in ("png", "svg"):
    fig.savefig(OUTPUT_DIR / f"volcano_RS_RC.{ext}", dpi=300)
plt.show()
print("Proteins passing volcano criteria:", len(selected))

## 7. Tornado plot — proteins selected from the volcano plot

This figure preserves the original directional tornado-plot design. Up to ten most negative and ten most positive **volcano-selected** proteins are shown, with healthy-enriched proteins to the left and cancer/cachexia-enriched proteins to the right. An optional second figure displays source q-values when available.

In [ ]:
negative = selected.loc[selected["log2FC"] < 0].nsmallest(
    TORNADO_TOP_PER_GROUP, "log2FC"
)
positive = selected.loc[selected["log2FC"] > 0].nlargest(
    TORNADO_TOP_PER_GROUP, "log2FC"
)
tornado = pd.concat([negative, positive]).drop_duplicates().sort_values("log2FC")
tornado.to_csv(OUTPUT_DIR / "tornado_volcano_selected_proteins.csv", index=False)
if tornado.empty:
    print("No proteins pass the volcano criteria; tornado plot not generated.")
else:
    fig, ax = plt.subplots(figsize=(12, max(6, len(tornado)*0.4)))
    values = tornado["log2FC"].to_numpy()
    ax.barh(np.arange(len(values)), values,
            color=np.where(values < 0, "#1f77b4", "#d62728"), edgecolor="black")
    ax.set_yticks(np.arange(len(values)), tornado["Description_clean"].str.slice(0, 65))
    ax.axvline(0, color="black", linewidth=1)
    ax.set_xlabel("log2 fold change (RC / RS)")
    ax.set_title("Tornado plot — volcano-selected proteins")
    fig.tight_layout()
    for ext in ("png", "svg"):
        fig.savefig(OUTPUT_DIR / f"tornado_volcano_selected.{ext}", dpi=300)
    plt.show()
    if "q Value" in tornado and tornado["q Value"].notna().any():
        fig, ax = plt.subplots(figsize=(12, max(6, len(tornado)*0.4)))
        ax.barh(np.arange(len(values)), values,
                color=np.where(values < 0, "#1f77b4", "#d62728"))
        ax.set_yticks(np.arange(len(values)), tornado["Description_clean"].str.slice(0, 55))
        ax.axvline(0, color="black")
        ax2 = ax.twinx()
        ax2.set_ylim(ax.get_ylim())
        ax2.set_yticks(np.arange(len(values)))
        ax2.set_yticklabels([
            f"{v:.1e}" if pd.notna(v) else "NA" for v in tornado["q Value"]
        ])
        ax2.set_ylabel("Source q-value")
        ax.set_xlabel("log2 fold change (RC / RS)")
        ax.set_title("Volcano-selected proteins with adjusted q-values")
        fig.tight_layout()
        for ext in ("png", "svg"):
            fig.savefig(OUTPUT_DIR / f"tornado_with_qvalues.{ext}", dpi=300)
        plt.show()

## 8. Tornado plot — the 20 most abundant proteins overall

Rank **unique proteins** by their combined mean abundance across the 12 individual preparations (6 RS + 6 RC). Select the **top 20 proteins overall**, not 10 from each group. Show **20 horizontal rows**; each row contains the RS abundance mirrored left and the RC abundance right, allowing direct comparison for the same protein. Negative horizontal coordinates are a plotting convention only, not negative protein abundance or fold changes.

In [ ]:
# Top 20 UNIQUE proteins overall, ranked by mean abundance across all 24 original injections.
TOP_ABUNDANT_PROTEINS = 20
abundance_table = analysis[["Description_clean", "Mean_RS", "Mean_RC"]].copy()
if "Accession" in analysis.columns:
    abundance_table["Accession"] = analysis["Accession"].astype(str)
for col in ["Mean_RS", "Mean_RC"]:
    abundance_table[col] = pd.to_numeric(abundance_table[col], errors="coerce")

# Means of 12 individual injections per group: each group contributes equally to the ranking.
abundance_table["Overall_mean_abundance"] = abundance_table[["Mean_RS", "Mean_RC"]].mean(axis=1)
abundance_table = abundance_table.loc[
    abundance_table["Overall_mean_abundance"].gt(0)
].copy()
# If multiple rows identify the same accession, retain its most abundant entry.
if "Accession" in abundance_table.columns:
    abundance_table = abundance_table.sort_values("Overall_mean_abundance", ascending=False)
    abundance_table = abundance_table.drop_duplicates(subset="Accession", keep="first")

top20 = abundance_table.nlargest(TOP_ABUNDANT_PROTEINS, "Overall_mean_abundance").copy()
if top20.empty:
    raise ValueError("No positive protein abundances available for the top-20 tornado plot.")
print(f"Tornado plot: {len(top20)} unique proteins, with RS and RC shown on each row.")
top20.to_csv(OUTPUT_DIR / "top20_most_abundant_proteins_overall.csv", index=False)

fig, ax = plt.subplots(figsize=(14, max(9, len(top20) * 0.48)))
y = np.arange(len(top20))
rs = top20["Mean_RS"].clip(lower=0).fillna(0).to_numpy(dtype=float)
rc = top20["Mean_RC"].clip(lower=0).fillna(0).to_numpy(dtype=float)
ax.barh(y, -rs, height=0.76, color="#1f77b4", label="Healthy serum (RS)")
ax.barh(y, rc, height=0.76, color="#d62728", label="Cancer/cachexia serum (RC)")
ax.axvline(0, color="black", linewidth=1)
ax.invert_yaxis()  # Most abundant protein at the top.
labels = [str(name)[:65] for name in top20["Description_clean"]]
ax.set_yticks(y, labels=labels)
ax.tick_params(axis="y", labelsize=8)
ax.set_xlabel("Mean protein abundance (RS mirrored left; RC right)")
ax.set_title("Top 20 most abundant proteins overall — RS vs RC")
from matplotlib.ticker import FuncFormatter
ax.xaxis.set_major_formatter(FuncFormatter(lambda value, pos: f"{abs(value):,.0f}"))
ax.legend(loc="lower right", frameon=False)
fig.tight_layout()
for ext in ("png", "svg"):
    fig.savefig(OUTPUT_DIR / f"tornado_top20_most_abundant_overall.{ext}",
                dpi=300, bbox_inches="tight")
plt.show()


## 9. Separate relative-abundance table: percentages, bubble plot, and paired bars

The original notebook additionally imports a table with `RS`, `RC`, `RS %`, and `RC %` and plots relative protein abundance. These plots are **not interchangeable with the volcano-selected plots**, so they are preserved as a separate analysis. If the table is not available, leave `RELATIVE_ABUNDANCE_FILE = None` and skip this section.

In [ ]:
# Set LOAD_RELATIVE_ABUNDANCE = True to select the second rat-serum CSV interactively.
LOAD_RELATIVE_ABUNDANCE = RELATIVE_ABUNDANCE_FILE is not None
if LOAD_RELATIVE_ABUNDANCE:
    rel = pd.read_csv(Path(RELATIVE_ABUNDANCE_FILE), low_memory=False)
    rel.columns = rel.columns.str.strip()
    required = ["Description", "RS", "RC", "RS %", "RC %"]
    missing = [col for col in required if col not in rel.columns]
    if missing:
        raise KeyError(f"Missing columns in relative-abundance table: {missing}")
    for col in required[1:]:
        rel[col] = pd.to_numeric(
            rel[col].astype(str).str.replace(",", ".", regex=False), errors="coerce"
        )
    rel["Protein"] = rel["Description"].astype(str).str.split(
        "OS=", regex=False
    ).str[0].str.strip()
    rel["log2FC"] = np.log2((rel["RC"]+1)/(rel["RS"]+1))
    top = pd.concat([
        rel.nlargest(TOP_RELATIVE_PROTEINS, "RC %"),
        rel.nlargest(TOP_RELATIVE_PROTEINS, "RS %")
    ]).drop_duplicates(subset="Protein")
    top.to_csv(OUTPUT_DIR / "top_relative_abundance.csv", index=False)
    long = top.melt(id_vars=["Protein"], value_vars=["RS %", "RC %"],
                    var_name="Group", value_name="Percentage")
    fig, ax = plt.subplots(figsize=(9, max(6, len(top)*0.35)))
    sns.scatterplot(data=long, x="Group", y="Protein", size="Percentage",
                    hue="Percentage", sizes=(35, 450), palette="viridis", ax=ax)
    ax.set_title("Relative protein abundance (%) — bubble plot")
    fig.tight_layout()
    for ext in ("png", "svg"):
        fig.savefig(OUTPUT_DIR / f"relative_abundance_bubble.{ext}", dpi=300)
    plt.show()
    fig, ax = plt.subplots(figsize=(10, max(6, len(top)*0.4)))
    y = np.arange(len(top))
    ax.barh(y-0.18, top["RS %"], height=0.36, label="RS")
    ax.barh(y+0.18, top["RC %"], height=0.36, label="RC")
    ax.set_yticks(y, top["Protein"].str.slice(0, 60))
    ax.invert_yaxis()
    ax.set_xlabel("Relative abundance (%)")
    ax.set_title("Top protein abundance — RS vs RC")
    ax.legend()
    fig.tight_layout()
    for ext in ("png", "svg"):
        fig.savefig(OUTPUT_DIR / f"relative_abundance_bars.{ext}", dpi=300)
    plt.show()
else:
    print("Relative-abundance CSV not supplied. Optional percentage plots skipped.")

## 10. Alternative protein filters and export

The source notebook tests multiple independent criteria: source q-value below 0.05, absolute log2FC at least 1, intensity at least 1e6, and less stringent exploratory thresholds. These are exported as **separate lists** and must not be conflated with the volcano definition.

In [ ]:
filters = {
    "volcano_p005_fc15": analysis["Volcano_Significant"],
    "absolute_log2fc_ge_1": analysis["log2FC"].abs() >= 1,
}
if "q Value" in analysis:
    filters["qvalue_lt_005"] = analysis["q Value"] < 0.05
if "Intensity" in analysis:
    filters["log2fc_ge_1_intensity_ge_1e6"] = (
        (analysis["log2FC"].abs() >= 1) & (analysis["Intensity"] >= 1e6)
    )
    filters["log2fc_ge_05_intensity_ge_1e5"] = (
        (analysis["log2FC"].abs() >= 0.5) & (analysis["Intensity"] >= 1e5)
    )
with pd.ExcelWriter(OUTPUT_DIR / "protein_selection_filters.xlsx", engine="openpyxl") as writer:
    for name, mask in filters.items():
        chosen = analysis.loc[mask.fillna(False)].copy()
        chosen.to_excel(writer, sheet_name=name[:31], index=False)
        print(f"{name}: {len(chosen)} proteins")
print("All available figures and tables saved to:", OUTPUT_DIR.resolve())